# A3BX3 band gaps: small data, honest validation

This notebook tells the story. **It computes nothing.** Every number is read from a committed
report that `python run_all.py` regenerates:

- `reports/metrics_v2.json`
- `reports/uncertainty_v2.json`
- `reports/explain_v2.json`
- `reports/mlp_v2.json`
- `data/a3bx3_family_metrics.json` (v1)

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
def load(rel):
    return json.loads((ROOT / rel).read_text(encoding="utf-8"))

M = load("reports/metrics_v2.json")
U = load("reports/uncertainty_v2.json")
E = load("reports/explain_v2.json")
N = load("reports/mlp_v2.json")
V1 = load("data/a3bx3_family_metrics.json")

def ci(e):
    return f"{e['MAE']:.3f} [{e['MAE_CI95'][0]:.3f}, {e['MAE_CI95'][1]:.3f}]"
print("reports loaded:", M["version"], "| environment:", M["environment"]["scikit-learn"])

reports loaded: v2.0-dev | environment: 1.9.1


## 1. The problem

**Goal.** Predict the GGA-PBE band gap of A3BX3 inverse-perovskite pnictogen halides:
A = Mg/Ca/Sr/Ba, B = P/As/Sb/Bi, X = F/Cl/Br/I.

**Why these compounds.** Sr3BiX3 is a candidate lead-free absorber.

**Why the obvious shortcut fails.** The obvious shortcut is to train on the large ABX3 perovskite
literature. But A3BX3 (3:1:3) is a different structural class from ABX3 (1:1:3).

## 2. Why the structural family matters more than the model

In v1, a model trained on ABX3 halides missed Sr3BiX3 by about **0.85 eV**. That figure is quoted
in the v1 README and is not stored in a metrics file.

A model restricted to the A3BX3 family, trained on 23 literature rows, landed within about
0.16 eV. The same question stopped being an extrapolation and became an interpolation.

In [ ]:
print("v1 family GPR on Sr3BiX3, MAE:", V1["sr3bix3_validation"]["MAE"], "eV")
print("v1 LOO / LOBO MAE:", V1["loo"]["MAE"], "/", V1["lobo"]["MAE"], "eV  (history; see section 3)")

v1 family GPR on Sr3BiX3, MAE: 0.16 eV
v1 LOO / LOBO MAE: 0.1512 / 0.1458 eV  (history; see section 3)


## 3. Data, and the corrections that v1 needed

Week 3 added a literature harvest of 37 sources. Checking every v1 row against its source found
real errors:

- **Mg3BiI3 and Mg3BiBr3 carried HSE06 gaps labelled as PBE** (0.867 and 1.626 eV instead of
  0.224 and 1.071 eV).
- Several rows cite a paper that does not contain their value.

The fixes live in `data/raw/literature/corrections.csv`. The v1 file itself is never edited.
The label is the **median** of the no-SOC PBE values for each formula.

In [ ]:
for d in ("primary", "verified_only", "plus_unverified", "soc"):
    c = M["datasets"][d]["curate"]
    print(f"{d:<16} {c['n_formulas']:>3} formulas  {c['n_values']:>3} values  "
          f"high_conflict {c['n_high_conflict']}  preprint-only {c['n_preprint_only']}")
print("K1 checkpoint:", M["k1"])

primary           39 formulas   57 values  high_conflict 3  preprint-only 8
verified_only     31 formulas   45 values  high_conflict 2  preprint-only 0
plus_unverified   42 formulas   61 values  high_conflict 3  preprint-only 8
soc               21 formulas   26 values  high_conflict 1  preprint-only 8
K1 checkpoint: {'pass': True, 'primary_formulas': 39, 'threshold': 35}


## 4. Honest validation

**How every score is produced:**

- Every score is out-of-fold, next to four named baselines on the **same folds**.
- Splits come from explicit grouped splitters.
- Every transform sits inside a `Pipeline`.
- Hyperparameters are chosen by nested CV.

**The headline split.** It is **leave-one-B-family-out**: a whole pnictogen chemistry is held
out at once.

![parity](../reports/figures/parity_lobo_gpr_physics9.png)

In [ ]:
print(f"{'set':<14}{'split':<6}{'gpr_physics9':<24}{'mean':<24}{'ridge_physics9':<24}best baseline")
for d in ("primary", "verified_only"):
    for s, r in M["datasets"][d]["results"].items():
        m = r["models"]
        print(f"{d:<14}{s:<6}{ci(m['gpr_physics9']):<24}{ci(m['mean']):<24}"
              f"{ci(m['ridge_physics9']):<24}{r['best_baseline']} {m[r['best_baseline']]['MAE']:.3f}")
k2 = U["k2"]["datasets"]["primary"]
print("\nK2:", k2["verdict"], "| skill vs mean", k2["vs_mean"]["skill"], k2["vs_mean"]["skill_CI95"],
      "| vs best baseline", k2["best_baseline"], k2["vs_best_baseline"]["skill"], k2["vs_best_baseline"]["skill_CI95"])

set           split gpr_physics9            mean                    ridge_physics9          best baseline
primary       gkf   0.094 [0.075, 0.115]    0.522 [0.405, 0.648]    0.359 [0.277, 0.455]    ridge_magpie 0.238
primary       loao  0.505 [0.377, 0.634]    0.619 [0.499, 0.743]    0.598 [0.467, 0.734]    ridge_magpie 0.496
primary       lobo  0.111 [0.080, 0.144]    0.526 [0.410, 0.652]    0.334 [0.251, 0.438]    ridge_magpie 0.198
primary       loo   0.090 [0.069, 0.115]    0.521 [0.404, 0.647]    0.350 [0.265, 0.449]    ridge_magpie 0.225
verified_only gkf   0.127 [0.101, 0.153]    0.520 [0.376, 0.674]    0.370 [0.272, 0.479]    ridge_magpie 0.319
verified_only loao  0.371 [0.234, 0.535]    0.584 [0.443, 0.736]    0.741 [0.529, 0.985]    halide_rule 0.499
verified_only lobo  0.137 [0.098, 0.177]    0.530 [0.395, 0.677]    0.427 [0.316, 0.558]    ridge_magpie 0.288

K2: PASS | skill vs mean 0.788 [0.717, 0.841] | vs best baseline ridge_magpie 0.439 [0.144, 0.63]


Two things temper the headline:

- The primary LOBO MAE (about 0.11 eV) is at the **label noise floor** of 0.1–0.2 eV, which is
  the code-to-code spread for a single compound.
- Against the best baseline, the lower CI bound of the skill is 0.14. That is below the 30 %
  K2 bar.

## 5. Uncertainty

**How the intervals are built.** Every interval is fitted on the training fold only:

- the GPR's σ, raw and rescaled by inner CV
- CV+ and jackknife+ (MAPIE), around the GPR and around the baselines

**Where the guarantee holds.** Exchangeability, and with it the conformal guarantee, holds for
GroupKFold. It does **not** hold when a whole chemistry is held out.

![reliability](../reports/figures/uncertainty_reliability_primary.png)

In [ ]:
methods = ["gpr_sigma", "gpr_sigma_scaled", "gpr_physics9_jackknife_plus", "ridge_physics9_cvplus", "mean_cvplus"]
print(f"{'method':<30}" + "".join(f"{s:>16}" for s in U["coverage"]["primary"]))
for mth in methods:
    row = ""
    for s, S in U["coverage"]["primary"].items():
        e = S["methods"][mth]["0.90"]
        row += f"{e['coverage']:>8.2f} ({e['mean_width']:.2f})"
    print(f"{mth:<30}{row}")
print("\ncoverage at nominal 90 % (mean width, eV); exchangeability:", U["exchangeability"])

method                                     gkf            loao            lobo             loo
gpr_sigma                         0.78 (0.35)    0.77 (1.90)    0.67 (0.34)    0.85 (0.33)
gpr_sigma_scaled                  0.93 (0.66)    1.00 (2.79)    0.90 (0.75)    0.92 (0.45)
gpr_physics9_jackknife_plus       0.95 (0.48)    0.21 (0.42)    0.87 (0.52)    0.92 (0.46)
ridge_physics9_cvplus             0.92 (1.78)    0.69 (1.67)    0.97 (2.21)    0.90 (1.79)
mean_cvplus                       0.91 (2.30)    0.85 (2.34)    0.90 (2.39)    0.92 (2.41)

coverage at nominal 90 % (mean width, eV); exchangeability: {'gkf': 'holds: shuffled grouped KFold, test formulas a random subset', 'loao': 'violated (whole A-site held out)', 'lobo': 'violated (whole B-family held out)', 'loo': 'approximately holds: each formula left out once'}


## 6. What the model learned, tested against physics written down beforehand

The expectations were pre-registered and committed **before** any SHAP value existed, in
`reports/explain_expectations.md`.

**The primary test.** Swap one element and watch the prediction. This holds up even though
descriptors are correlated inside the family. SHAP dependence slopes are secondary evidence.

![substitution](../reports/figures/explain_substitution.png)

In [ ]:
print("expectations commit:", E["expectations"]["commit"])
for model, r in E["physics_check"].items():
    print(f"\n{model}")
    for k, v in r.items():
        verdict = v.get("verdict") if isinstance(v, dict) and "verdict" in v else v
        print(f"  {k:<42} {verdict}")
g = E["models"]["gpr_physics9"]["splits"]["gkf"]
print("\nGPR top features:", g["top10"][:5], "| rank stability (Kendall tau):", g["rank_stability"]["kendall_tau_all_features"])
print("Mg - Ca substitution (no prior):", g["substitution"]["A:Mg-Ca"]["mean"], "eV")

expectations commit: 76f6425f6a40e0dd779c5ff0b298b24c8bdd6194

gpr_physics9
  E1 halide I<Br<Cl<F                        CONFIRMED
  E1 secondary SHAP slopes                   {'chiX>0': 'UNCLEAR', 'rX<0': 'UNCLEAR'}
  E2 pnictogen P>...>Bi (decisive P-Bi)      CONFIRMED
  E2 secondary SHAP slopes                   {'chiB>0': 'UNCLEAR', 'rB<0': 'CONFIRMED'}
  E3 A-site Ca>Sr>Ba (decisive Ca-Ba)        CONFIRMED
  E3 secondary SHAP slopes                   {'chiA (reported)': -0.3623, 'rA<0': 'CONFIRMED'}
  E4 dchi_BX slope>0                         UNCLEAR
  E6 A and X groups > B group (SHAP)         UNCLEAR

rf_magpie
  E1 halide I<Br<Cl<F                        CONFIRMED
  E2 pnictogen P>...>Bi (decisive P-Bi)      CONFIRMED
  E3 A-site Ca>Sr>Ba (decisive Ca-Ba)        CONFIRMED

xgb_magpie
  E1 halide I<Br<Cl<F                        CONFIRMED
  E2 pnictogen P>...>Bi (decisive P-Bi)      CONFIRMED
  E3 A-site Ca>Sr>Ba (decisive Ca-Ba)        CONFIRMED

GPR top features: ['rA', 'dchi

**Findings:**

- **The element-level trends hold for all three models:**
  - halide I < Br < Cl < F
  - pnictogen P > Bi
  - A-site Ca > Sr > Ba
- **Single-descriptor SHAP signs are often UNCLEAR.** The GPR's ARD kernel switches off chiX and
  dchi_BX and routes the halide effect through rX. With correlated descriptors, attributions to
  one feature at a time are not interpretable.
- **An unregistered observation.** Mg compounds are predicted 0.3–0.4 eV below Ca compounds. It
  is a hypothesis for DFT, not a result.

![beeswarm](../reports/figures/explain_beeswarm_gpr_physics9.png)

## 7. Does a neural network help? (the November milestone)

A small PyTorch MLP was trained on the same features and the same folds. It uses a nested grid,
early stopping on an inner validation split and an average of 5 seeds. The decision rule, and
the expected answer of "no", were committed before training
(`reports/mlp_expectations.md`).

![learning curve](../reports/figures/mlp_learning_curve.png)

In [ ]:
v = N["verdict"]
print("MLP beats XGB:", v["MLP beats XGB"], "| MLP beats GPR:", v["MLP beats GPR"], "| rule commit:", N["expectations"]["commit"][:7])
for pair, o in v["outcomes"].items():
    print(f"  {pair:<30} {o}")
P = N["datasets"]["primary"]
for s in ("lobo", "loao"):
    m = P[s]["models"]
    print(f"{s}: " + "  ".join(f"{k} {m[k]['MAE']:.3f}" for k in ("mlp_physics9", "mlp_magpie", "xgb_physics9", "xgb_magpie", "gpr_physics9", "mean")))
lc = N["learning_curve"]["models"]
print("learning curve (MAE at 50/75/100 % of each training fold):")
for k, v_ in lc.items():
    print(f"  {k:<14}", " -> ".join(f"{v_[f]['MAE']:.3f}" for f in sorted(v_)))

MLP beats XGB: NO | MLP beats GPR: NO | rule commit: e153c25
  mlp_magpie vs gpr_physics9     {'loao': 'LOSS', 'lobo': 'LOSS'}
  mlp_magpie vs mean             {'loao': 'LOSS', 'lobo': 'WIN'}
  mlp_magpie vs xgb_magpie       {'loao': 'LOSS', 'lobo': 'LOSS'}
  mlp_physics9 vs gpr_physics9   {'loao': 'TIE', 'lobo': 'LOSS'}
  mlp_physics9 vs mean           {'loao': 'TIE', 'lobo': 'WIN'}
  mlp_physics9 vs xgb_physics9   {'loao': 'LOSS', 'lobo': 'TIE'}
lobo: mlp_physics9 0.182  mlp_magpie 0.204  xgb_physics9 0.170  xgb_magpie 0.151  gpr_physics9 0.111  mean 0.526
loao: mlp_physics9 0.652  mlp_magpie 1.294  xgb_physics9 0.489  xgb_magpie 0.434  gpr_physics9 0.505  mean 0.619
learning curve (MAE at 50/75/100 % of each training fold):
  gpr_physics9   0.209 -> 0.115 -> 0.094
  mlp_physics9   0.320 -> 0.199 -> 0.138
  xgb_magpie     0.271 -> 0.210 -> 0.167
  xgb_physics9   0.287 -> 0.233 -> 0.190


**What it shows:**

- With about 31 training rows per fold, the MLP ties XGBoost at best and loses to the GPR.
- The learning curve slopes down for every model. More compounds would help all of them.
- A different architecture would not change the answer: the set is data-limited, not
  model-limited.

## 8. Limits (the binding list is MODEL_CARD.md §7)

- These are PBE gaps without SOC, not experimental ones.
- The label noise floor is 0.1–0.2 eV.
- n = 39, and 21 % of the formulas come only from a preprint.
- **No intervals and no trust for an unseen A-site.** LOAO conformal coverage is 21–36 % at
  nominal 90 %.
- The model covers the cubic 4 × 4 × 4 family only.